In [9]:
import pandas as pd
import bz2
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import os

# Çözüm: punkt_tab eklendi
nltk.download('punkt')
nltk.download('punkt_tab') 
nltk.download('stopwords')

# 2. Veri Okuma ve Temizleme Fonksiyonu
def read_and_clean_bz2(file_path, num_lines=50000):
    labels = []
    texts = []
    
    # İngilizce stopword'leri bir kümeye alıyoruz (işlem hızını artırır)
    stop_words = set(stopwords.words('english'))
    
    print(f"{num_lines} satır okunuyor ve NLP ön işlemlerinden geçiriliyor. Lütfen bekleyin...")
    
    # bz2 formatındaki sıkıştırılmış dosyayı aç ve satır satır oku
    with bz2.BZ2File(file_path, 'r') as f:
        for i, line in enumerate(f):
            if i >= num_lines:
                break
                
            # Byte verisini string'e çevir
            line = line.decode('utf-8')
            
            # Etiket (__label__1 veya __label__2) ve yorum metnini birbirinden ayır
            label = line.split(' ')[0]
            raw_text = line[len(label)+1:].strip()
            
            # Etiketi modelin anlayacağı formata çevir: __label__2 -> 1 (Pozitif), __label__1 -> 0 (Negatif)
            sentiment = 1 if label == '__label__2' else 0
            
            # --- METİN TEMİZLEME (NLP PREPROCESSING) ADIMLARI ---
            clean_txt = raw_text.lower() # Küçük harfe çevir
            clean_txt = clean_txt.translate(str.maketrans('', '', string.punctuation)) # Noktalama işaretlerini kaldır
            tokens = word_tokenize(clean_txt) # Kelimelere ayır (Tokenization)
            tokens = [word for word in tokens if word not in stop_words] # Stopword'leri çıkar
            final_text = " ".join(tokens) # Kelimeleri tekrar temiz bir cümleye birleştir
            
            labels.append(sentiment)
            texts.append(final_text)
            
    # Temizlenmiş listeleri bir Pandas DataFrame'ine dönüştür
    return pd.DataFrame({'review': texts, 'sentiment': labels})

# 3. Dinamik Dosya Yolu Oluşturma (Her bilgisayarda çalışması için)
current_dir = os.getcwd()
file_path = os.path.join(current_dir, 'data', 'archive (4)', 'train.ft.txt.bz2')

# 4. Fonksiyonu çalıştır ve 50.000 satırlık verimizi al
df = read_and_clean_bz2(file_path, num_lines=50000)

# 5. Sonuçları Görüntüle
print("\n--- İlk 5 Satır ---")
display(df.head())

print("\n--- Sınıf Dağılımı (1: Pozitif, 0: Negatif) ---")
print(df['sentiment'].value_counts())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\asusr\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\asusr\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asusr\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


50000 satır okunuyor ve NLP ön işlemlerinden geçiriliyor. Lütfen bekleyin...

--- İlk 5 Satır ---


,review,sentiment
0,stuning even nongamer sound track beautiful pa...,1
1,best soundtrack ever anything im reading lot r...,1
2,amazing soundtrack favorite music time hands i...,1
3,excellent soundtrack truly like soundtrack enj...,1
4,remember pull jaw floor hearing youve played g...,1



--- Sınıf Dağılımı (1: Pozitif, 0: Negatif) ---
sentiment
1    25506
0    24494
Name: count, dtype: int64
